# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này được thiết kế để chạy trực tiếp thông qua **Google Colab Extension trong VS Code** hoặc mở trên web **Google Colab**.

### 💡 Thông tin về gói Google One AI Premium (AI One Pro) & GPU Colab:
- Gói **Google One AI Premium** cung cấp sẵn Compute Units cho Google Colab, cho phép bạn truy cập GPU tốc độ cao mà không lo hết quota 30h của Kaggle:
  - **T4 GPU** (15GB VRAM): Hoàn toàn đủ cho Qwen3-4B-Instruct với `--max-ctx 8192` và `--gpu-mem 0.88`.
  - **L4 GPU** (24GB VRAM): Tốc độ suy luận nhanh gấp ~2-3 lần T4, VRAM thoải mái cho cả Planner + Dialect Mapper.
  - **A100 GPU** (40GB/80GB VRAM): Tốc độ cực nhanh cho toàn bộ pipeline.

### ⚙️ Cách kết nối trong VS Code với Google Colab Extension:
1. Cài extension **Google Colab** từ VS Code Marketplace.
2. Mở file này trong VS Code -> Bấm vào nút chọn Kernel ở góc trên bên phải -> Chọn **Connect to Google Colab** (hoặc `Ctrl+Shift+P` -> gõ `Colab: Connect to Colab Runtime`).
3. Đăng nhập tài khoản Google (tài khoản có gói AI One Pro) -> Chọn phần cứng có GPU (**T4** hoặc **L4**).
4. Chạy tuần tự các cell bên dưới.

In [1]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

Sat Nov  7 09:41:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
google-adk 2.7.1 requires op

In [3]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os
from pathlib import Path

if not os.path.exists('ZAIC_2026'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git
    %cd ZAIC_2026
else:
    %cd ZAIC_2026
    !git fetch origin

!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

Cloning into 'ZAIC_2026'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (414/414), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 414 (delta 172), reused 391 (delta 150), pack-reused 0 (from 0)
Receiving objects: 100% (414/414), 1.09 MiB | 37.19 MiB/s, done.
Resolving deltas: 100% (172/172), done.
/content/ZAIC_2026
branch 'planner/vuong' set up to track 'origin/planner/vuong'.
Switched to a new branch 'planner/vuong'
From https://github.com/Bamboohoccode/ZAIC_2026
 * branch            planner/vuong -> FETCH_HEAD
Already up to date.

Commit mới nhất trên nhánh planner/vuong:
b258d42 (HEAD -> planner/vuong, origin/planner/vuong) [notebooks] them colab_run_planner.ipynb ho tro chay tren Google Colab / VS Code extension


In [ ]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Nhanh & Trực tiếp trong 1 giây)
import os, glob
from pathlib import Path

os.makedirs('data', exist_ok=True)

# 1. Nếu có file zip data.zip ở /content hoặc MyDrive -> giải nén tức thì
zip_paths = ['/content/data.zip', 'data.zip', '/content/drive/MyDrive/data.zip']
for z in zip_paths:
    if os.path.exists(z):
        print(f'📦 Đang giải nén {z}...')
        !unzip -q -o "{z}" -d .
        break

# 2. Kiểm tra các đường dẫn phổ biến (tránh rglob đệ quy toàn bộ Google Drive gây treo máy)
direct_candidates = [
    Path('data/train'),
    Path('/content/data/train'),
    Path('/content/train'),
    Path('/content/drive/MyDrive/data/train'),
    Path('/content/drive/MyDrive/ZAIC_2026/data/train'),
    Path('/content/drive/MyDrive/ZAIC_2026/train'),
    Path('/content/drive/MyDrive/train'),
    Path('../data/train'),
]

found_train = None
for p in direct_candidates:
    if (p / 'train.jsonl').exists():
        found_train = p.resolve()
        break

# 3. Quét nhanh tối đa 2 cấp thư mục nếu chưa thấy
if not found_train:
    for base in ['/content', '/content/drive/MyDrive']:
        if os.path.exists(base):
            for cand in glob.glob(f'{base}/*/train.jsonl') + glob.glob(f'{base}/*/*/train.jsonl'):
                found_train = Path(cand).parent.resolve()
                break
        if found_train:
            break

if found_train and str(found_train) != str(Path('data/train').resolve()):
    print(f'✅ Đã tìm thấy dữ liệu tại: {found_train}')
    if not os.path.exists('data/train'):
        os.symlink(str(found_train), 'data/train')
    pub_cand = found_train.parent / 'public_test'
    if (pub_cand / 'public_test.jsonl').exists() and not os.path.exists('data/public_test'):
        os.symlink(str(pub_cand), 'data/public_test')

if os.path.exists('data/train/train.jsonl'):
    print('✅ Thư mục data/train đã sẵn sàng:')
    !ls -l data/train | head -n 5
else:
    print('⚠️ Chưa tìm thấy train.jsonl!')
    print('Cách xử lý nhanh nhất: Chạy lệnh symlink trực tiếp đến nơi bạn để data:')
    print('Ví dụ: !ln -s "/content/drive/MyDrive/ĐƯỜNG_DẪN/train" data/train')

In [ ]:
# 5. Chạy Champion planner_v4 kèm Safe Hash Guard trên 300 mẫu Dev
# Pipeline đã tích hợp tự động Safe Hash Guard, tự cắt tỉa context chống tràn vLLM, và tự động lưu log ra file <out>.log
import os, torch
os.makedirs('runs', exist_ok=True)

# Tự động tối ưu gpu-mem theo VRAM thực tế:
vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
gpu_mem = 0.88 if vram < 20 else 0.90
print(f'GPU VRAM: {vram:.1f} GB -> Cấu hình --gpu-mem {gpu_mem}')

!python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem {gpu_mem} \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --out runs/dev_v4_guarded.jsonl

In [ ]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
!python eval/scorer.py --sub runs/dev_v4_guarded.jsonl

In [ ]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')

In [ ]:
# 8. (Tuỳ chọn - Trụ cột 3) Chạy thử nghiệm Dialect Mapper (mapper_v1.txt + Qwen3-0.6B)
# !python -m lifemate.pipeline \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper Qwen/Qwen3-0.6B \
#     --mapper-prompt prompts/gepa/vuong/mapper_v1.txt \
#     --gpu-mem 0.78 \
#     --mapper-gpu-mem 0.10 \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --out runs/dev_v4_mapper_v1.jsonl
#
# !python eval/scorer.py --sub runs/dev_v4_mapper_v1.jsonl